# draw market — `mrkt_favourite` segments → model on the most profitable segment

`goals_foragst` space, draw market, on the shared `evaluation.model_check` framework. Two parts:

1. **Raw segment profitability** (descriptive, in-sample) — bet the draw on *every* match in each
   `mrkt_favourite` segment (home / away / balanced), no model. Shows where draws are underpriced.
2. **Model on the best segment** — the SEGMENTER picks, per training fold, the single `mrkt_favourite`
   segment with the best draw edge, and labels only that segment; the model then bets the draw there
   where it beats implied by `buffer`, under the three gates.

This is the **rule-ish** kind of segmenter: instead of clustering, it selects among a handful of
existing categories. Unlike the original notebook (which chose the best segment in-sample, once), the
choice here is made on the TRAIN fold of each split, so it never peeks at the held-out data.

In [ ]:
# --- SETUP -------------------------------------------------------------------
# Load the wide ABT for one league group and pull in the model-check framework. Everything that
# never changes between checks (folds, ROI scoring, the honest per-fold engine, the three gates,
# the registry write) lives in `evaluation.model_check`; this notebook only defines the parts that
# DO change — the space, the target, the segmenter, and the model.
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import (
    run_check, evaluate, collect_bets, pooled_roi, roi_grouped,
    season_folds, expanding_folds,
)

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [ ]:
# --- SPACE (modifiable) ------------------------------------------------------
# The bet-signal model sees the four running goals for/against features (the goals_foragst space).
FEATURES = [
    "hmt_season_goals_for_avg", "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg", "awt_season_goals_agst_avg",
]

In [ ]:
# --- TARGET (modifiable) -----------------------------------------------------
OUTCOME = "t_draw_flg"              # 1 when the match ended in a draw
IMPLIED = "b365_draw_impl"          # implied draw prob we bet against = 1 / b365 draw odds
MRKT_IMPLIED = "mrkt_draw_impl"     # market-consensus implied draw prob, for profitability only
SEGMENT_COL = "mrkt_favourite"      # the category we segment on: home / away / balanced

In [ ]:
# --- ABT FILTERING -----------------------------------------------------------
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [SEGMENT_COL, IMPLIED, MRKT_IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}")
print(f"segments ({SEGMENT_COL}): {matches[SEGMENT_COL].value_counts().to_dict()}")

In [ ]:
# --- SEGMENTER (modifiable) --------------------------------------------------
# Same contract: segmenter(train) -> labeler ; labeler(rows) -> Series, NaN = don't bet.
# This one does NOT cluster — it ranks the existing `mrkt_favourite` categories by their draw edge on
# the TRAIN fold and keeps only the single best one. A clean example of a rule/selection segmenter:
# the region is defined by an existing column, and "which segment" is the only thing chosen on train.
def favourite_best_segmenter(min_matches=50):
    def segmenter(train):
        edges = {}
        for seg, sub in train.groupby(SEGMENT_COL):
            if len(sub) < min_matches:
                continue
            edges[seg] = sub[OUTCOME].mean() - sub[MRKT_IMPLIED].mean()   # train draw edge
        best = max(edges, key=edges.get) if edges else None
        def labeler(rows):
            if best is None:
                return pd.Series(np.nan, index=rows.index)
            return rows[SEGMENT_COL].where(rows[SEGMENT_COL] == best)     # only the best segment
        return labeler
    return segmenter

segmenter = favourite_best_segmenter()

In [ ]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: given the space features, predict the probability of the target outcome.
# Today it is polynomial-expanded features (squares + pairwise products), standardised, then logistic
# regression. To try a different method (random forest, XGBoost, ...) swap this one factory — the
# only contract the framework needs is an sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Part 1 — raw draw profitability per segment (descriptive, in-sample)
Bet the draw on every match in each segment (no model). ROI under implied-probability staking, b365
odds. Just a map of where draws look underpriced — not evidence, and not what the gates below judge.

In [ ]:
print("raw draw ROI per mrkt_favourite segment (bet draw on all matches, b365):")
for seg in sorted(matches[SEGMENT_COL].unique()):
    sub = matches[matches[SEGMENT_COL] == seg]
    roi = pooled_roi(sub[IMPLIED], sub[OUTCOME].astype(int))
    frame = pd.DataFrame({"implied": sub[IMPLIED], "won": sub[OUTCOME].astype(int), "fold": sub["season"]})
    per_season = roi_grouped(frame, "fold")
    print(f"  {seg:9s} ROI {roi:+.4f}  ({len(sub)} matches)  "
          + "  ".join(f"{s}:{v:+.3f}" for s, v in per_season.items()))

## Summary — model on the train-best segment, three gates
The segmenter picks the best `mrkt_favourite` segment per training fold; the model bets the draw
there. Row logged to `model_checks/results.md`.

In [ ]:
run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
          segmenter=segmenter, make_model=make_model, buffer=BUFFER,
          source="goals_foragst_draw_favourite_best_lr", space="goals_foragst", target="draw",
          abt_filter="major, gn>8, dropna", segmentation="mrkt_favourite",
          segment_filter="best train-edge segment")